# Cropy Qwen STT 강건성 평가

지금까지 Qwen 정확도(97.0%)는 사람이 쓴 문장으로 잰 값이다. 같은 검증 1,160문장을 **합성 음성(MeloTTS) → 실제 STT(SenseVoice)**로
받아 적은 문장으로 다시 재서, 실제 사용에서 얼마나 떨어지는지 본다. 라벨은 그대로다.

| 실행 | 입력 |
|---|---|
| `_typed` | 원문 (기준) |
| `_clean` · `_snr10` · `_snr5` | STT 결과(용어 보정 후, 문장부호 제거) · 조용 / 잡음 10dB / 잡음 5dB |
| `_clean_punct` | 조용, 문장부호를 지우지 않음 (학습 데이터에 문장부호가 없어서 앱에서 지워야 하는지 확인) |
| `_clean_raw` | 조용, 용어 보정 전 문장 (보정이 도움이 되는지 확인) |

- 모델: 웹 묶음 `web_onnx/2b/bundle`의 q4f16 (원본과 300건 중 297건 같은 답).
- 런타임: **A100**(L4도 됨). 실행 6번 × 1,160건, 약 30분. 결과는 `cropy_qwen_v2/stt_eval/`.
- 먼저 Drive `cropy_qwen_v2/stt_eval/`에 `val_stt.jsonl`를 올린다(바탕화면에 복사해 둠).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
DRIVE_DIR = Path('/content/drive/MyDrive/cropy_qwen_v2')
DATA_FILE = DRIVE_DIR / 'qwen_v2c_data.jsonl.xz'
DATA_SHA256 = '756723fbd79a025d0a6e4129c8bcfcf9d71bef09efb0789c386bc39080539516'
OUT = DRIVE_DIR / 'stt_eval'
TEXTS = OUT / 'val_stt.jsonl'
TEXTS_SHA256 = '814074af810c77087c6986782ac9b8ff8e01ef1f94a6c42404b7410c8fa2108f'
BUNDLE_DRIVE = DRIVE_DIR / 'web_onnx' / '2b' / 'bundle'
WORK = Path('/content/stt_eval')
WORK.mkdir(parents=True, exist_ok=True)
RUNS = [('_typed', None, None, False), ('_clean', 'clean', 'stt_text', True), ('_snr10', 'snr10', 'stt_text', True), ('_snr5', 'snr5', 'stt_text', True), ('_clean_punct', 'clean', 'stt_text', False), ('_clean_raw', 'clean', 'stt_raw', True)]
assert hashlib.sha256(DATA_FILE.read_bytes()).hexdigest() == DATA_SHA256, '데이터 파일이 노트북과 다릅니다'
assert TEXTS.exists(), f'{TEXTS}를 먼저 올리세요'
assert hashlib.sha256(TEXTS.read_bytes().replace(bytes([13, 10]), bytes([10]))).hexdigest() == TEXTS_SHA256, 'STT 결과 파일이 노트북과 다릅니다'
BUNDLE = WORK / 'bundle'
if not (BUNDLE / 'cropy_prompt.json').exists():
    shutil.copytree(BUNDLE_DRIVE, BUNDLE, dirs_exist_ok=True)
print('묶음', sum(f.stat().st_size for f in BUNDLE.rglob('*') if f.is_file()) / 2**30, 'GB')

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu==1.31.1', 'transformers==5.19.0', 'tokenizers==0.23.2'])
data = (DRIVE_DIR / 'stt_eval_scripts' / 'eval_web_onnx.py').read_bytes().replace(bytes([13, 10]), bytes([10]))
assert hashlib.sha256(data).hexdigest() == 'd90615498f0d45557bbc578f2733a78ef14bd34db7fbe04368d33a22cdaa0715', 'eval_web_onnx.py가 노트북과 다릅니다(다시 올리세요)'
(WORK / 'eval_web_onnx.py').write_bytes(data)

def run(*argv):
    p = subprocess.Popen([sys.executable, '-u', *map(str, argv)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'{argv[0]} 실패'

## 평가 (실행마다 1,160건, 끝난 실행은 건너뜀)

In [ ]:
for tag, cond, field, strip in RUNS:
    if (OUT / f'web_q4f16{tag}_metrics.json').exists():
        print('이미 있음', tag); continue
    args = ['--bundle', BUNDLE, '--data', DATA_FILE, '--data-sha', DATA_SHA256, '--dtype', 'q4f16', '--out-dir', WORK,
            '--sim-n', -1, '--device', 'cuda', '--tag', tag]
    if cond:
        args += ['--texts', TEXTS, '--condition', cond, '--text-field', field]
    if strip:
        args += ['--strip-punct']
    run(WORK / 'eval_web_onnx.py', *args)
    for name in (f'web_q4f16{tag}_metrics.json', f'web_q4f16{tag}_cases.jsonl'):
        shutil.copy(WORK / name, OUT / name)

## 요약

In [ ]:
rows = []
for tag, cond, field, strip in RUNS:
    m = json.loads((OUT / f'web_q4f16{tag}_metrics.json').read_text())
    rows.append((tag, m))
    print(f"{tag:14s} 형식 {m['format_valid']:.4f} 기능 {m['function']:.4f} 인자까지 {m['exact']:.4f}")
base = dict(rows)['_typed']
print()
print(f"{'함수':28s}" + ''.join(f'{t:>14s}' for t, _ in rows))
for fn in base['per_function']:
    print(f'{fn:28s}' + ''.join(f"{m['per_function'][fn]['exact']:14.3f}" for _, m in rows))
(OUT / 'summary.json').write_text(json.dumps({t: {k: m[k] for k in ('format_valid', 'function', 'exact', 'per_function')} for t, m in rows},
                                             ensure_ascii=False, indent=2))